# Notebook 2 — PII & Secret Protection (Solo Exercise)
### Module: Enterprise AI Security & Guardrails · 2 of 5

In Notebook 1 you stopped attackers from making InternalAssist *say* things
it shouldn't. This notebook is about a quieter failure: InternalAssist
leaking personal data **without any attacker at all** — just because an
employee record flowed through it and nobody masked it.

Two places that leak matters:
- **On the way out** — a Social Security number or card number appearing in
  a chat response.
- **On the way in to a vector store** — raw PII getting embedded and indexed,
  where it lives indefinitely and resurfaces in future retrievals (the
  pipeline Notebook 3 builds). Anonymising *before* embedding is the only
  reliable fix; you can't un-embed a number later.

**This is a solo exercise.** Cells marked **🔧 YOUR TURN** have a `# TODO`
for you to complete. The mechanics (finding spans, masking without
corrupting offsets) are given to you in `security_utils/pii.py` — what you
decide is *which* entities to catch and how to *prove* nothing leaks. Run
`pytest tests/test_pii.py -v` to check yourself.


## 0 · Setup

`security_utils.pii` gives you a recognizer that prefers Microsoft Presidio
(NER + patterns) and silently falls back to a regex-only backend if
Presidio's spaCy model isn't installed — so this notebook runs anywhere.
Check which backend you got: if it says `regex`, names and addresses won't
be detected (only structured IDs), which is itself a lesson in why you don't
rely on regex alone for PII.


In [ ]:
import sys, os, json, re
sys.path.insert(0, os.getcwd())

from security_utils.pii import get_recognizer, mask_text, PiiSpan, DEFAULT_PII_PATTERNS
from security_utils.logging_utils import AuditLogger

recognizer = get_recognizer()
audit = AuditLogger(path="logs/pii_audit.jsonl")

print(f"PII backend: {recognizer.backend}")
print(f"Default regex entity types: {list(DEFAULT_PII_PATTERNS)}")


## 1 · See the leak

A realistic helpdesk record. Watch what an unprotected pipeline would put
into a chat response or an embedding.


In [ ]:
EMPLOYEE_RECORD = (
    "Ticket #4471 from Jane Doe (jane.doe@northwind.com). "
    "Verified identity with SSN 123-45-6789. Callback number 555-123-4567. "
    "Corporate card on file 4111 1111 1111 1111. "
    "Last login from 10.0.0.42."
)
print(EMPLOYEE_RECORD)
print("\nDetected spans:")
for s in recognizer.detect(EMPLOYEE_RECORD):
    print(f"  {s.entity_type:14s} [{s.start}:{s.end}] -> {EMPLOYEE_RECORD[s.start:s.end]!r}")


## 2 · 🔧 YOUR TURN — mask before exposure

Write `redact()` so it returns the record with **every** detected PII span
replaced by its `<ENTITY_TYPE>` placeholder. You already have the two pieces
you need: `recognizer.detect(text)` and `mask_text(text, spans)`.


In [ ]:
def redact(text: str) -> str:
    # TODO: detect PII spans in `text`, then return the masked version.
    # Hint: this is two function calls.
    raise NotImplementedError

# --- check it ---
masked = redact(EMPLOYEE_RECORD)
print(masked)
assert "123-45-6789" not in masked, "SSN leaked!"
assert "jane.doe@northwind.com" not in masked, "email leaked!"
assert "4111 1111 1111 1111" not in masked, "card leaked!"
print("\n✅ no structured PII survived")


## 3 · 🔧 YOUR TURN — measure redaction *coverage*

"It looked masked" is not a security guarantee. Coverage = the fraction of
PII spans that actually got masked. Write `coverage()` to return a number in
`[0.0, 1.0]`: of all spans the recognizer finds in the original text, how
many no longer appear as raw substrings in the redacted output?

This is the metric `pytest tests/test_pii.py` asserts on — the exercise's
real deliverable.


In [ ]:
def coverage(original: str, redacted: str) -> float:
    spans = recognizer.detect(original)
    if not spans:
        return 1.0
    # TODO: count how many of the detected raw substrings are GONE from
    # `redacted`, and divide by the total number of spans.
    raise NotImplementedError

cov = coverage(EMPLOYEE_RECORD, redact(EMPLOYEE_RECORD))
print(f"Redaction coverage: {cov:.0%}")
assert cov == 1.0, "Some PII slipped through — fix redact() or your patterns."
print("✅ full coverage")


## 4 · Anonymise before embedding

The critical ordering rule: **mask first, embed second.** Here's a tiny
simulation of an ingestion step — the only correct version redacts before
the text would ever reach `embed()`.


In [ ]:
def embed(text: str) -> list[float]:
    # Stand-in for a real embedding model. The point isn't the vector --
    # it's that whatever string we pass here gets stored forever.
    return [float(len(text))]

def ingest_document(text: str) -> dict:
    safe_text = redact(text)            # <-- mask BEFORE embedding
    audit.log(event="pii_redaction", session_id="ingest",
              verdict="redacted",
              detail={"n_spans": len(recognizer.detect(text))})
    return {"stored_text": safe_text, "vector": embed(safe_text)}

record = ingest_document(EMPLOYEE_RECORD)
print("stored_text:", record["stored_text"])
assert "123-45-6789" not in record["stored_text"]
print("\n✅ what got stored carries no raw SSN")


## 5 · 🔧 YOUR TURN (stretch) — add a custom entity

The regex backend doesn't know about Northwind's internal employee IDs,
which look like `EMP-` followed by 5 digits (e.g. `EMP-04471`). Add a
pattern so they get masked too. Then confirm coverage stays at 100%.


In [ ]:
# TODO: add an "EMPLOYEE_ID" entry to this dict matching EMP-#####
CUSTOM_PATTERNS = dict(DEFAULT_PII_PATTERNS)
# CUSTOM_PATTERNS["EMPLOYEE_ID"] = re.compile(r"...")

from security_utils.pii import RegexRecognizer
custom = RegexRecognizer(patterns=CUSTOM_PATTERNS)

sample = "Escalating ticket for EMP-04471 (jane.doe@northwind.com)."
spans = custom.detect(sample)
masked = mask_text(sample, spans)
print(masked)
assert "EMP-04471" not in masked, "employee ID leaked — add the pattern above"
print("✅ custom entity masked")


## 5b · Graduate your solution so pytest can check it

Once the asserts above pass, write your two functions to a module. The test
suite imports from here — same "notebook prototype → real module" pattern as
Notebook 1. (If you skip this, `pytest tests/test_pii.py` skips with a
reminder.)


In [ ]:
%%writefile security_utils/pii_solution.py
"""Your completed NB2 redaction logic. Written from the notebook."""
from security_utils.pii import get_recognizer, mask_text

_recognizer = get_recognizer()


def redact(text: str) -> str:
    return mask_text(text, _recognizer.detect(text))


def coverage(original: str, redacted: str) -> float:
    spans = _recognizer.detect(original)
    if not spans:
        return 1.0
    gone = sum(1 for s in spans if original[s.start:s.end] not in redacted)
    return gone / len(spans)


## 6 · Audit trail & wrap-up

Every redaction was logged to `logs/pii_audit.jsonl` — a durable record that
ingestion *happened*, without storing the PII itself (the log keeps a count,
never the values). In a real LangSmith setup you'd also tag the trace; here
the local audit log is the same idea, vendor-independent.

```bash
pytest tests/test_pii.py -v
```

**Next — Notebook 3 (Guided Lab): RAG Poisoning.** You'll build the retrieval
pipeline these redacted documents flow into, then watch an attacker poison it
with an adversarial document, and add source-validation guardrails.


In [ ]:
print(f"{len(audit.read_all())} redaction events logged to logs/pii_audit.jsonl")
